In [2]:
# manual run
import logging
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

from demo_and_analysis.benchmark.run import get_all_query_ids
from misc.dataset.dataset_tables_dict import get_dataset_name
from misc.dataset.query_gen_factory import get_query_gen
from pipeline.cpp_runner.compiler_utils import make_compiler
from pipeline.git_snapshotter import GitSnapshotter
from pipeline.tools.run import RunTool
from pipeline.tools.validate.query_validator_class import QueryValidator
from prepare_repo.load_snapshot_and_prepare import prepare_repo_and_load_snapshot
from utils.logging_and_reporting.logger import setup_logging
from utils.logging_and_reporting.wandb_api_helper import wandb_retrieve_metrics_for_run
from utils.utils import DBStorage, get_disk_db_dir

setup_logging(logging.DEBUG)

cache_dir = Path("/mnt/labstore/bespoke_olap/cache")
workspace_dir = Path.cwd() / "tmp"
# workspace_dir = Path.cwd().parent.parent / "output"
base_parquet_dir = "/mnt/labstore/bespoke_olap/tpch_parquet"

benchmark = "tpch"
query_list = get_all_query_ids(benchmark)

snapshot = None  # "06e5fc67b0e6dd6a800b998d4b9b7b76fdd5a1c0"
wandb_id = None  # "yj67splq"
prepare = None  # "optim"  # base / optim / mt
db_storage = DBStorage.SSD

CACHE_REPO = "git://c01/bespoke_cache.git"
if snapshot is not None or wandb_id is not None:
    if snapshot is not None and wandb_id is not None:
        raise ValueError("Cannot specify both snapshot and wandb_id")
    if wandb_id is not None:
        statistics, config, hist = wandb_retrieve_metrics_for_run(
            benchmark="tpch", run_id=wandb_id, output_hist=True
        )
        snapshot = statistics["code/snapshot_hash"]
    assert snapshot is not None, (
        "Snapshot must be specified if wandb_id is not specified"
    )

    snapshotter = GitSnapshotter(
        cache_repo=CACHE_REPO,
        working_dir=workspace_dir,
        extra_gitignore=[],
        do_not_snapshot=True,
    )

    assert prepare is not None, (
        "Prepare must be specified if snapshot or wandb_id is specified"
    )
    prepare_repo_and_load_snapshot(
        snapshotter=snapshotter,
        snapshot=snapshot,
        prepare=prepare,
        benchmark=benchmark,
        query_list=query_list,
        cache_path=cache_dir,
        db_storage=db_storage,
    )

disk_db_dir, bespoke_dir = get_disk_db_dir(db_storage, workspace_dir)

gen_query_fn = get_query_gen(benchmark)
query_validator = QueryValidator(
    benchmark=benchmark,
    gen_query_fn=gen_query_fn,
    sf_list=[1, 2, 20],
    parquet_path=base_parquet_dir,
    wandb_pin_worker=True,
    all_query_ids=query_list,
    num_random_query_instantiations=10,
    query_cache_dir=cache_dir / "query_cache",
    validate_cache_dir=None,
    workspace_path=workspace_dir,
    git_snapshotter=None,
    runtime_tracker=None,
    do_not_cache=True,
    db_storage=db_storage,
    run_umbra_as_well=False,
    disk_db_dir=disk_db_dir,
)

db_engine = RunTool(
    cwd=workspace_dir,
    dataset_name=get_dataset_name(benchmark),
    base_parquet_dir=base_parquet_dir,
    run_stats_collector=None,
    query_validator=query_validator,
    parallelism=False,
    core_ids=[0, 1, 2, 3, 4, 5, 6, 7],
    db_storage=db_storage,
    bespoke_storage_dir=bespoke_dir,
    memory_budget_mb=50 * 1024 if db_storage != DBStorage.IN_MEMORY else None,
    compiler=make_compiler(
        cwd=workspace_dir,
        db_storage=db_storage,
        untracked_cpp_runner_content="",
    ),
)


2026-05-22 06:39:23 INFO:pipeline.tools.validate.query_validator_class:Initializing query cache with pre-generated instantiations...
2026-05-22 06:39:26 INFO:pipeline.tools.validate.query_cache:Query pre-generation complete (loaded from cache or executed again)
2026-05-22 06:39:26 DEBUG:pipeline.cpp_runner.compiler:pkg-config --cflags arrow parquet
2026-05-22 06:39:26 DEBUG:pipeline.cpp_runner.compiler:pkg-config --libs arrow parquet


In [4]:
msg, metrics, trace_output = db_engine.run(
    scale_factor=50,
    optimize=True,
    query_id=[
        "1",
        "2",
        "3",
        "4",
        "5",
        "6",
        "7",
        "8",
        "9",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
    ],
    trace_mode=False,
    echo_output=True,
)

print(f"==============\nOutput msg:\n==============\n{msg}")
# print(f"==============\nMetrics:\n==============\n{metrics}")
print(f"==============\nTrace output:\n==============\n{trace_output}")

2026-05-22 06:44:34 INFO:pipeline.tools.run:Run with: query_id=['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22'] scale_factor=50 self.dataset_name='tpch' trace_mode=False optimize=True self.base_parquet_dir='/mnt/labstore/bespoke_olap/tpch_parquet' num_threads=1 mem_limit=51200
2026-05-22 06:44:34 INFO:pipeline.tools.run:Deleting existing result-csv files (66 files).


Output msg:
Scale factor 50 not available in query validator (not prepared). Available scale factors: [1, 2, 20]
Trace output:
None
